# Database overview

Exploração reproduzível e exclusivamente read-only do banco canônico do desafio.

# 1. Setup

O notebook importa o pacote instalado e abre o SQLite distribuído em modo read-only.

In [ ]:
from __future__ import annotations

import hashlib
import platform
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from bracis_jusbrasil.database import connect_database, get_database_path

DATABASE_PATH = get_database_path()
connection = connect_database(DATABASE_PATH)

print({
    'python': platform.python_version(),
    'sqlite': sqlite3.sqlite_version,
    'database': str(DATABASE_PATH),
})

In [ ]:
def query_frame(sql: str, params: tuple[object, ...] = ()) -> pd.DataFrame:
    return pd.read_sql_query(sql, connection, params=params)


def text_preview(text: str, limit: int = 420) -> str:
    compact = ' '.join(text.split())
    return compact[:limit] + ('…' if len(compact) > limit else '')


def text_context(text: str, position: int, limit: int = 260) -> str:
    if position < 0:
        return text_preview(text, limit)
    start = max(0, position - limit // 3)
    end = min(len(text), position + 2 * limit // 3)
    return text_preview(text[start:end], limit)

# 2. SQLite schema

Objetos, campos e índices que estruturam o acervo.

In [ ]:
objects = query_frame(
    "SELECT type, name, tbl_name, sql FROM sqlite_master ORDER BY type, name"
)
display(objects.assign(sql=objects['sql'].fillna('').map(lambda value: text_preview(value, 180))))

display(query_frame("PRAGMA table_info(documentos)"))
display(query_frame("PRAGMA index_list(documentos)"))

fts_schema = query_frame(
    "SELECT sql FROM sqlite_master WHERE name = 'documentos_fts'"
)
display(fts_schema.assign(sql=fts_schema['sql'].map(lambda value: text_preview(value, 420))))
display(query_frame("PRAGMA table_info(documentos_fts)"))

# 3. Visão geral

Contagens e distribuições do conteúdo canônico.

In [ ]:
total = query_frame("SELECT COUNT(*) AS total_registros FROM documentos")
por_natureza = query_frame("SELECT natureza, COUNT(*) AS total FROM documentos GROUP BY natureza ORDER BY natureza")
por_tipo = query_frame("SELECT tipo, COUNT(*) AS total FROM documentos GROUP BY tipo ORDER BY tipo")
por_tribunal = query_frame("SELECT tribunal, COUNT(*) AS total FROM documentos WHERE tribunal IS NOT NULL GROUP BY tribunal ORDER BY tribunal")
intervalo_anos = query_frame("SELECT MIN(ano) AS primeiro_ano, MAX(ano) AS ultimo_ano FROM documentos WHERE ano IS NOT NULL")
por_ano = query_frame("SELECT ano, COUNT(*) AS total FROM documentos WHERE ano IS NOT NULL GROUP BY ano ORDER BY ano")

display(total)
display(por_natureza)
display(por_tipo)
display(por_tribunal)
display(intervalo_anos)
display(por_ano)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
por_natureza.plot.bar(x='natureza', y='total', legend=False, ax=axes[0], title='Registros por natureza')
por_ano.plot.bar(x='ano', y='total', legend=False, ax=axes[1], title='Registros por ano')
fig.tight_layout()
plt.show()

# 4. Exemplos de registros

Os trechos são truncados para evitar a impressão de documentos inteiros.

In [ ]:
def examples_for_natureza(natureza: str) -> pd.DataFrame:
    examples = query_frame(
        '''
        SELECT documento_id, id, tribunal, ano, relator, natureza, tipo, texto
        FROM documentos
        WHERE natureza = ?
        ORDER BY documento_id
        LIMIT 3
        ''',
        (natureza,),
    )
    examples['inicio_do_texto'] = examples.pop('texto').map(lambda value: text_preview(value, 420))
    return examples

for natureza in ('acordao', 'sumula', 'dispositivo'):
    print(natureza)
    display(examples_for_natureza(natureza))

# 5. Cabeçalhos por tribunal

A comparação visual ajuda a observar classe, número, UF e relator sem criar um parser definitivo.

In [ ]:
for tribunal in ('STF', 'STJ', 'TSE', 'TST', 'STM'):
    headers = query_frame(
        '''
        SELECT documento_id, id, ano, relator, texto
        FROM documentos
        WHERE natureza = 'acordao' AND tribunal = ?
        ORDER BY documento_id
        LIMIT 3
        ''',
        (tribunal,),
    )
    headers['inicio_do_acordao'] = headers.pop('texto').map(lambda value: text_preview(value, 900))
    print(tribunal)
    display(headers)

# 6. Súmulas

Os cinco registros permitem verificar visualmente se o número aparece no texto disponível.

In [ ]:
sumulas = query_frame(
    "SELECT documento_id, id, tribunal, texto FROM documentos WHERE natureza = 'sumula' ORDER BY tribunal, documento_id"
)
sumulas['trecho_relevante'] = sumulas.pop('texto').map(lambda value: text_preview(value, 900))
display(sumulas)

# 7. Dispositivos legais

Os treze registros mostram como artigos e diplomas foram armazenados no acervo.

In [ ]:
dispositivos = query_frame(
    "SELECT documento_id, id, texto FROM documentos WHERE natureza = 'dispositivo' ORDER BY documento_id"
)
dispositivos['trecho_relevante'] = dispositivos.pop('texto').map(lambda value: text_preview(value, 700))
display(dispositivos)

# 8. FTS5

A busca a seguir é apenas exploratória: ela ilustra a diferença entre uma ocorrência no cabeçalho e uma simples menção no corpo.

In [ ]:
def search_fts(query: str, occurrence: str, limit: int = 5) -> pd.DataFrame:
    rows = query_frame(
        '''
        SELECT d.documento_id, d.id, d.tribunal, d.natureza, d.texto
        FROM documentos_fts
        JOIN documentos AS d ON d.rowid = documentos_fts.rowid
        WHERE documentos_fts MATCH ?
        LIMIT ?
        ''',
        (query, limit),
    )
    rows['posicao_aproximada'] = rows['texto'].map(lambda value: value.casefold().find(occurrence.casefold()))
    rows['contexto'] = rows.apply(lambda row: text_context(row['texto'], row['posicao_aproximada']), axis=1)
    rows['inicio_do_documento'] = rows['texto'].map(lambda value: text_preview(value, 420))
    return rows.drop(columns='texto')

display(search_fts('"1.996.496"', '1.996.496'))

# O segundo exemplo retorna menções em posições distintas: a tabela permite
# comparar o início do acórdão com o contexto, sem inferir uma resolução.
display(search_fts('"2.110"', '2.110', limit=10))

In [ ]:
fts_2110 = search_fts('\"2.110\"', '2.110', limit=10)
display(fts_2110)

fig, ax = plt.subplots(figsize=(8, 4))
(fts_2110.sort_values('posicao_aproximada')
    .plot.barh(x='documento_id', y='posicao_aproximada', legend=False, ax=ax, color='#4472c4'))
ax.set_title('Distribuição das posições dos resultados FTS5')
ax.set_xlabel('posição aproximada no texto')
ax.set_ylabel('documento')
fig.tight_layout()
plt.show()

# 9. Duplicatas

Os hashes são calculados apenas em memória e jamais persistidos no SQLite.

In [ ]:
textos = query_frame("SELECT documento_id, id, tribunal, texto FROM documentos")
textos['hash'] = textos['texto'].map(lambda value: hashlib.sha256(value.encode('utf-8')).hexdigest())
duplicados = textos[textos['hash'].duplicated(keep=False)].copy()
grupos = (
    duplicados.groupby('hash', as_index=False)
    .agg(
        registros=('documento_id', 'size'),
        documento_ids=('documento_id', lambda values: ', '.join(map(str, values))),
        ids_canonicos=('id', lambda values: ', '.join(map(str, values))),
        tribunais=('tribunal', lambda values: ', '.join(sorted({str(value) for value in values}))),
    )
    .sort_values(['registros', 'hash'], ascending=[False, True])
)
print({'grupos_duplicados': len(grupos), 'registros_envolvidos': len(duplicados)})
display(grupos.head(5))

# 10. Findings

## Fatos

- A exploração confirma 1.018 registros, distribuídos em três naturezas.
- A tabela FTS5 está disponível para consultas textuais.
- Os cabeçalhos dos acórdãos variam entre tribunais.

## Dúvidas abertas

- Como diferenciar, em uma futura solução, um cabeçalho canônico de uma mera menção ao mesmo processo no corpo de outro documento?

In [ ]:
connection.close()
print('Conexão read-only encerrada.')